# 🥐 Where should we open our bakery?

---

We have **three candidate locations** for a new bakery. We want to know which one sits in a neighbourhood that most closely resembles where **existing bakeries already succeed**.

## The Approach
1. Find all existing bakeries in the area using **Overture Maps** (open global dataset, queried directly from cloud storage)
2. Generate a **catchment profile** around each bakery using the **Ahmad's custom tool**
4. Do the same for each candidate location
5. Find the candidate whose catchment profile **most closely matches the benchmark**


| | |
|---|---|
| **Cloud-native** | No data downloaded. Overture Maps queried directly from Amazon S3. Census data queried spatially on demand. |
| **Two APIs, one notebook** | `arcpy` and the ArcGIS API for Python work side by side — each doing what it does best |
| **Modular** | Walking area generation and Census querying live in a reusable Python module, keeping this notebook focused on the analysis |
| **Reproducible** | Change the bounding box, the place category, or the candidate locations — re-run and get a new answer |

---
## 1. Imports & Sign In



**Esri libraries**
- `arcpy` — Esri's geoprocessing engine. Used here for feature class operations and the Summarize Within tool
- `arcgis` — The ArcGIS API for Python. Handles portal authentication, hosted layers, and online services

**Open source / Python data science stack**
- `duckdb` — Runs SQL queries directly against cloud-hosted Parquet files on S3. 
- `pandas` — Tabular data manipulation throughout
- `matplotlib` — Charting

**Custom module**
- `Catchment profile tool` — A reusable Python module containing the walking area generation and Census OA querying logic. Keeping it separate means this notebook stays focused on the analysis, and the same routing/Census logic can be reused in other notebooks



In [3]:


# ── Esri ──────────────────────────────────────────────────────────────────────
import arcpy
from arcgis.gis import GIS
from arcgis.features import GeoAccessor, FeatureLayer
from arcgis.geometry import Point as AGPoint

# ── Open source data science stack ───────────────────────────────────────────
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── Custom module (walking area generation + Census OA querying) ──────────────

import importlib, site

module_code = open('/arcgis/home/Catchment_profile_tool.py').read()
with open('/opt/conda/lib/python3.13/site-packages/Catchment_profile_tool.py', 'w') as f:
    f.write(module_code)
from Catchment_profile_tool import generate_walking_areas, generate_walking_areas_from_df, get_occupation_profile

import warnings, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

arcpy.env.overwriteOutput = True

gis = GIS('home')
print(f"Signed in as: {gis.properties.user.username}")
print(f"DuckDB version: {duckdb.__version__}")

Signed in as: BBurgess@esriuk.com_EsriUK
DuckDB version: 1.3.2


---
## 2. Configuration



All analysis parameters live here. To run this for a different city, place type, or set of candidates — only this section needs editing.

The three **parameters** (`CANDIDATES_ITEM_ID`, `BBOX`, `PLACE_CATEGORY`) are tagged so this notebook can be published as a **custom ArcGIS tool** — users supply these values through a form interface without ever seeing the code.

In [ ]:
# ── Tool parameters ───────────────────────────────────────────────────────────
# These are exposed as inputs when this notebook is published as a custom tool.
# When running interactively, edit these values directly.
CANDIDATES_ITEM_ID = "a8f83b2ead264f1c8109a4495a475ece"          # ArcGIS Online item ID for candidate locations
BBOX               = "-2.994890, 53.402778, -2.973433, 53.413471" # Study area: west, south, east, north
PLACE_CATEGORY     = "bakery"                                      # Overture Maps place category to benchmark against

In [ ]:
# ── Parse BBOX string into individual coordinates ─────────────────────────────
# BBOX comes in as a comma-separated string (required for tool parameters)
xmin, ymin, xmax, ymax = [float(v.strip()) for v in BBOX.split(',')]

# ── Overture Maps release ─────────────────────────────────────────────────────
RELEASE = '2026-04-15.0'

# ── Candidate locations feature layer URL ─────────────────────────────────────
candidates_item = gis.content.get(CANDIDATES_ITEM_ID)
CANDIDATES_URL  = candidates_item.layers[0].url

# ── Census 2021 TS063 — Occupation by Output Area (ONS / Esri UK hosted) ──────
# Publicly accessible on ArcGIS Online — no download needed
OCC_URL = (
    'https://services.arcgis.com/qHLhLQrcvEnxjtPr/arcgis/rest/services/'
    'Pop_Census21_EW_LabourMarket_TS063_Occupation/FeatureServer/7'
)

# Count fields only (even-numbered = counts, odd = rates)
OCC_FIELDS = {
    'I38556D0': '1. Managers, directors & senior officials',
    'I38558D0': '2. Professional occupations',
    'I38560D0': '3. Associate professional & technical',
    'I38562D0': '4. Administrative & secretarial',
    'I38564D0': '5. Skilled trades',
    'I38566D0': '6. Caring, leisure & other service',
    'I38568D0': '7. Sales & customer service',
    'I38570D0': '8. Process, plant & machine operatives',
    'I38572D0': '9. Elementary occupations',
}

# ── Routing service parameters ────────────────────────────────────────────────
SA_BREAK_VALUES = '15'           # Walk time cutoff in minutes
SA_BREAK_UNITS  = 'Minutes'
SA_TRAVEL_MODE  = 'Walking Time'

print(f"Area of interest  : {xmin},{ymin} -> {xmax},{ymax}")
print(f"Place category    : {PLACE_CATEGORY}")
print(f"Candidates item   : {CANDIDATES_ITEM_ID}")
print("Configuration loaded.")

---
## 3. Build the Bakery Benchmark



**Overture Maps** is an open global dataset of 300+ million places, roads, and buildings maintained by a consortium including Microsoft, Meta, Amazon, and TomTom. It lives on Amazon S3 as Parquet files.

**DuckDB** lets us query it directly with SQL — the query runs against S3 and only the matching rows come back. We never download the full dataset.

Once we have the bakery locations, we pass them to the `Catchment profile tool` module which:
1. Calls the **ArcGIS Online routing service** to generate a 15-minute walking polygon around each bakery
2. Queries the **Census FeatureServer** to find which Output Areas fall within each polygon
3. Returns the median occupation profile across all those Output Areas


In [ ]:
# ── Step 1: Query bakeries from Overture Maps on S3 ───────────────────────────
# DuckDB connects directly to S3 — no data is downloaded to this notebook
conn = duckdb.connect()
for ext in ('spatial', 'httpfs'):
    conn.sql(f'INSTALL {ext}; LOAD {ext};')
conn.sql("SET s3_region='us-west-2'; SET enable_object_cache=true;")

bakeries_df = conn.sql(f"""
    SELECT id, names.primary AS name, ST_X(geometry) AS lon, ST_Y(geometry) AS lat
    FROM read_parquet(
        's3://overturemaps-us-west-2/release/{RELEASE}/theme=places/type=place/*.parquet',
        hive_partitioning=1
    )
    WHERE bbox.xmin >= {xmin} AND bbox.ymin >= {ymin}
      AND bbox.xmax <= {xmax} AND bbox.ymax <= {ymax}
      AND categories.primary = '{PLACE_CATEGORY}'
""").df()
print(f"Found {len(bakeries_df)} {PLACE_CATEGORY} locations")

# ── Step 2: Build SHAPE column (ArcGIS point geometry) ────────────────────────
bakeries_df['SHAPE'] = bakeries_df.apply(
    lambda r: AGPoint({'x': r.lon, 'y': r.lat, 'spatialReference': {'wkid': 4326}}), axis=1
)
bakeries_df.spatial.set_geometry('SHAPE')


# ── Step 3: Generate walking catchments + query Census OAs (via module) ────────
# generate_walking_areas_from_df — takes a dataframe (not a portal item)
# because bakeries come from Overture Maps, not ArcGIS Online
bakery_walking_sdf = generate_walking_areas_from_df(
    gis, bakeries_df[['name', 'SHAPE']],
    break_values=SA_BREAK_VALUES, break_units=SA_BREAK_UNITS, travel_mode=SA_TRAVEL_MODE
)
_, benchmark_pct, all_oas = get_occupation_profile(
    bakery_walking_sdf, OCC_URL, OCC_FIELDS, gis._con.token
)
print(f"Benchmark built from {len(all_oas)} OAs across {len(bakeries_df)} {PLACE_CATEGORY} locations")



# ── Step 4: Show this output on a map (woo maps!) ────────

m2 = gis.map()
m2.extent = {
    'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax,
    'spatialReference': {'wkid': 4326}
}

# Walking catchments — add as a FeatureCollection with explicit symbology
from arcgis.features import FeatureSet
from arcgis.map.renderers import SimpleRenderer

walk_fs = FeatureSet.from_dataframe(bakery_walking_sdf)
bakery_fs = FeatureSet.from_dataframe(bakeries_df)

m2.content.add(walk_fs, options={
    'renderer': {
        'type': 'simple',
        'symbol': {
            'type': 'esriSFS',
            'style': 'esriSFSSolid',
            'color': [161, 206, 227, 80],
            'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                        'color': [33, 102, 172, 180], 'width': 1}
        }
    }
})

m2.content.add(bakery_fs, options={
    'renderer': {
        'type': 'simple',
        'symbol': {
            'type': 'esriSMS',
            'style': 'esriSMSCircle',
            'color': [33, 102, 172, 230],
            'size': 8,
            'outline': {'color': [255, 255, 255, 200], 'width': 1}
        }
    }
})

m2

---
## 4. Profile the Candidate Locations



The candidate locations are stored as a **hosted feature layer on ArcGIS Online** — so we use `generate_walking_areas` (item ID version) rather than the dataframe version used for bakeries above.

The module does the same thing as Step 3 above:
- Routing service → 15-minute walking polygon per candidate
- Census FeatureServer → OAs within each polygon
- Median occupation profile per candidate

The key difference from the benchmark step is that here we get **individual profiles per candidate** rather than a single combined profile — so we can compare each one against the benchmark.

> **Note:** The routing service call consumes ArcGIS Online credits (approx. 0.5 credits per location).

In [ ]:
# ── Load candidate names and coordinates ──────────────────────────────────────
# We fetch these separately so we have the original names for the output table and map
candidate_layer = FeatureLayer(CANDIDATES_URL, gis=gis)
candidates_fs   = candidate_layer.query(out_sr=4326)

candidates_df = pd.DataFrame([{
    'name': f.attributes.get('name') or f.attributes.get('Name') or f'Candidate {i+1}',
    'lon' : f.geometry['x'],
    'lat' : f.geometry['y'],
} for i, f in enumerate(candidates_fs.features)])
print(f"Loaded {len(candidates_df)} candidate locations")

# ── Generate walking catchments + query Census OAs (via module) ───────────────
# generate_walking_areas — takes an item ID because candidates are hosted on ArcGIS Online
walking_sdf = generate_walking_areas(
    gis, CANDIDATES_ITEM_ID,
    break_values=SA_BREAK_VALUES, break_units=SA_BREAK_UNITS, travel_mode=SA_TRAVEL_MODE
)
candidate_profiles, _, _ = get_occupation_profile(
    walking_sdf, OCC_URL, OCC_FIELDS, gis._con.token
)
print(f"Profiles built for {len(candidate_profiles)} candidates")




# ── Map: candidate locations and their walking catchments ─────────────────────
candidates_df['SHAPE'] = candidates_df.apply(
    lambda r: AGPoint({'x': r.lon, 'y': r.lat, 'spatialReference': {'wkid': 4326}}), axis=1
)
candidates_df.spatial.set_geometry('SHAPE')

m3 = gis.map()
m3.extent = {
    'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax,
    'spatialReference': {'wkid': 4326}
}

# Walking catchment polygons in orange
m3.content.add(FeatureSet.from_dataframe(walking_sdf), options={
    'renderer': {
        'type': 'simple',
        'symbol': {
            'type': 'esriSFS',
            'style': 'esriSFSSolid',
            'color': [253, 174, 97, 80],
            'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                        'color': [215, 25, 28, 180], 'width': 1}
        }
    }
})

# Candidate points in red
m3.content.add(FeatureSet.from_dataframe(candidates_df), options={
    'renderer': {
        'type': 'simple',
        'symbol': {
            'type': 'esriSMS',
            'style': 'esriSMSDiamond',
            'color': [215, 25, 28, 230],
            'size': 10,
            'outline': {'color': [255, 255, 255, 200], 'width': 1}
        }
    }
})

m3

---
## 5. Find the Best Match



We compare each candidate's occupation profile to the benchmark using a **gap score** — the total percentage point difference across all nine occupation groups.

**Lower score = closer match to where bakeries already succeed.**



In [ ]:
# loc_num extracts the integer from strings like "Location 1" or "Candidate 2"
# so we can match routing service output back to original candidate names
def loc_num(s):
    try:
        return int(''.join(filter(str.isdigit, s)))
    except:
        return None

# Key the profiles by location number so they align with candidates_df
profile_by_num = {loc_num(k): v for k, v in candidate_profiles.items()}

# Gap score = sum of absolute percentage point differences across all occupation groups
candidates_df['distance'] = candidates_df.index.map(
    lambda i: (profile_by_num[i + 1] - benchmark_pct).abs().sum().round(1)
    if (i + 1) in profile_by_num else None
)

candidates_df = candidates_df.sort_values('distance').reset_index(drop=True)
best = candidates_df.iloc[0]

print("=== Results (lower score = closer match to existing bakery catchments) ===\n")
print(candidates_df[['name', 'distance']].to_string(index=False))
print(f"\n  Best match: {best['name']}  (gap score = {best['distance']} pp)")

---
## 6. Visualise



- The **dashed line at zero** is the bakery benchmark — a perfect match would sit flat on it
- Each coloured line is a candidate location
- **Above zero** = more of that occupation group than typical bakery neighbourhoods
- **Below zero** = less
- The **green band** (±5 percentage points) gives a visual sense of what "close enough" looks like

The chart shows not just *which* candidate wins but *why* — which specific occupation groups make one location a better or worse fit.

In [ ]:
%matplotlib inline
plt.close('all')   # clear any previously queued figures from earlier runs

# one colour per candidate
colors = ['#2166ac', '#d6604d', '#4dac26']
# one x position per occupation group
x = list(range(len(OCC_FIELDS)))

fig, ax = plt.subplots(figsize=(12, 5))

# plot each candidate as a line showing deviation from the benchmark
for (_, row), color in zip(candidates_df.iterrows(), colors):
    num  = loc_num(row['name'])   # match candidate name back to profile by number
    if num not in profile_by_num:
        continue
    diff  = (profile_by_num[num] - benchmark_pct).values   # deviation from benchmark per group
    label = f"{row['name']}  (gap: {row['distance']} pp)"
    ax.plot(x, diff, marker='o', linewidth=2, label=label, color=color)

# zero line = the benchmark itself
ax.axhline(0, color='black', linewidth=1, linestyle='--', label='Bakery benchmark')

# green band = within 5 percentage points — a visual guide for "close enough"
ax.fill_between(x, -5, 5, alpha=0.05, color='green', label='Within 5 pp of benchmark')

# x axis labels from OCC_FIELDS readable names, stripped of their number prefix
ax.set_xticks(x)
ax.set_xticklabels([v.split('.')[1].strip() for v in OCC_FIELDS.values()],
                   rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Difference from benchmark (percentage points)')
ax.set_title(
    f'15-min Walking Catchment vs Bakery Benchmark\n'
    f'Benchmark: {len(all_oas)} OAs from {len(bakeries_df)} {PLACE_CATEGORY} locations',
    fontsize=12
)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 7. Save Results to ArcGIS Online

The analysis results are published back to ArcGIS Online as hosted feature layers using `to_featurelayer` from the ArcGIS API for Python. 

In [ ]:
# Publish walking area polygons
published_wa = walking_sdf.spatial.to_featurelayer(
    title       = f"{SA_BREAK_VALUES}-Minute Walking Areas — Candidate Locations2",
    gis         = gis,
    tags        = "Service Area, Walking, Bakery",
    description = f"{SA_BREAK_VALUES}-minute {SA_TRAVEL_MODE} catchments for {len(candidates_df)} candidate locations."
)
print(f"Walking areas published : {published_wa.homepage}")

# Publish candidate scores with point geometry so they can be mapped
candidates_df['SHAPE'] = candidates_df.apply(
    lambda r: AGPoint({'x': r.lon, 'y': r.lat, 'spatialReference': {'wkid': 4326}}), axis=1
)
candidates_df['best_match'] = candidates_df['name'] == best['name']
candidates_df[['name', 'distance', 'best_match', 'SHAPE']].spatial.set_geometry('SHAPE')
published_res = candidates_df[['name', 'distance', 'best_match', 'SHAPE']].spatial.to_featurelayer(
    title = 'Candidate Location Occupation Match Scores',
    gis   = gis,
    tags  = "Bakery, Occupation, Census 2021, Candidate",
)
print(f"Results published       : {published_res.homepage}")

---
## Summary

| Step | What happened | How |
|---|---|---|
| Find bakeries | Queried 300M+ global places dataset | DuckDB → Overture Maps on S3 |
| Walking catchments | 15-min walk polygon per location | ArcGIS routing service via Python API |
| Occupation profiles | Census OAs within each catchment | Direct REST query to ONS FeatureServer |
| Scoring | Gap score vs benchmark | pandas |
| Publish | Results back to ArcGIS Online | ArcGIS API for Python |

**Nothing was downloaded. No desktop software was used. The entire analysis ran in a browser.**

To run this for a different city or place type — change `BBOX` and `PLACE_CATEGORY` in the configuration cell and re-run.